# **Procesamiento del Lenguaje Natural**
## *Práctica final - Detección de lenguaje ofensivo en redes sociales*

## Recursos a importar

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [17]:
# NLTK
import nltk
nltk.download('punkt')

# Divisor de textos en oraciones
from nltk.tokenize import sent_tokenize

# Tokenizador de palabras
from nltk.tokenize import TreebankWordTokenizer
from nltk.tokenize import WhitespaceTokenizer
from nltk.tokenize import SpaceTokenizer
from nltk.tokenize import WordPunctTokenizer

# Análisis morfológico
import spacy.cli
spacy.cli.download("es_core_news_sm")
import es_core_news_sm
nlp = es_core_news_sm.load()

# Recurso para acceder a archivos del sistema
import os

# Corpus de palabras vacias
from nltk.corpus import stopwords
nltk.download("stopwords")
spanish_stops = stopwords.words('spanish')

# Reducir palabras a su raíz
from nltk.stem.snowball import SnowballStemmer
stemmer = SnowballStemmer("spanish")

# Vector de características
from sklearn.feature_extraction.text import TfidfVectorizer

# Modelo SVM
from sklearn import svm

# Recursos para abrir ficheros de excel/xls
import pandas as pd
import xlrd 

# Recuros extras
import csv
import re

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
✔ Download and installation successful
You can now load the model via spacy.load('es_core_news_sm')
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


## Variables Globales
Definimos una serie de variables globales que serán compartidas por diferentes funciones

In [18]:
# Variables globales
model = None
vectorizer = None
listaComentariosEntrenar: list = []
listaComentariosTest: list = []
lexicon : set = set()

# Quitamos la palabra no de los stopWords
if('no' in spanish_stops):
  spanish_stops.remove('no')

# Clase de comentario
Los atributos de la clase son:
* **comment_id**: Identificador del comentario
* **comment**: Comentario
* **label**: Etiqueta con la que se ha clasificado
* **influencer_gender**: Género de la persona a la que va dirigido el comentario
* **media**: Plataforma en la que se ha publicado el mensaje

In [19]:
class Comentario:
  def __init__(self, comment_id, comment, label, influencer_gender, media):
    self.comment_id = comment_id
    self.comment = comment
    self.label = label
    self.influencer_gender = influencer_gender
    self.media = media

## Función de preocesar texto
Función que recibe como parámetros:
* **texto** : Texto que se quiere procesar.  

Los pasos de la función son:
  1. Pasar el texto a minúscula, quitar los signos de puntuación y las tildes
  2. Tokenizar el texto, extrayendo las palabras
  3. Quitar las palabras vacías
  4. Obtener las raíces de cada palabra/token extraído
  5. Devolver el texto procesado

In [20]:
def procesarTexto(texto):
  tk = WordPunctTokenizer() # Tokenizador a utilizar

  # Quitamos los signos de puntuación del texto y pasamos a minúscula
  textoPuntuacion = re.sub(r'[^\w\s]','',texto.lower())
  # Quitamos las tildes del comentario
  textoPuntuacion = textoPuntuacion.translate(textoPuntuacion.maketrans("áàäéèëíìïòóöùúü", "aaaeeeiiiooouuu"))
  # Tokenizamos el texto
  palabras = tk.tokenize(str(textoPuntuacion))

  # Quitamos las palabras vacías
  palabrasNoVacias = [word for word in palabras if word not in spanish_stops]

  # Reducimos las palabras de toda la lista a su raíz
  listaRaices = []
  for palabra in palabrasNoVacias:
    listaRaices.append(stemmer.stem(palabra))
  
  # Devolvemos una cadena con el texto procesado
  return " ".join(listaRaices)

## Función para obtener contenido TSV y almacenarlo en una lista
Función que recibe como parámetros:
* **rutaArchivo** : Ruta del archivo excel que contiene los datos
* **listaArchivos**: Lista donde se van a almacenar los comentarios

Los pasos de la función son:
  1. Abrimos el fichero que contiene los comentarios
  2. Extraemos cada comentario línea a línea y lo almacenamos en la lista pasada

In [21]:
def obtenerDocumentosTSV(rutaArchivo, listaArchivos):
  # Comprobamos si existe el archivo pasado
  try:
    # Abrimos el archivo 
    f = open(rutaArchivo, "r", errors="ignore")

    # Separamos cada línea
    read_tsv = csv.reader(f, delimiter="\t")
    
    for r in read_tsv:
      # Leemos todas las lineas menos la primera que son los títulos
      comentario = Comentario(r[0],r[1],r[2],r[3],r[4])

      listaArchivos.append(comentario)

    # Eliminamos la primera línea que contiene los títulos
    listaArchivos.pop(0)
    print("El número de comentarios es de: " + str(len(listaArchivos)))
    f.close()
  except FileNotFoundError:
    print("No existe el archivo " + rutaArchivo)

## Función para obtener el lexicón
Función que recibe como parámetros:
* **rutaArchivo**: Ruta del archivo *.txt*, que contiene el lexicón

Los pasos a realizar por la función son:
1. Abrir el fichero pasado como parámetro
2. Leer cada línea del fichero
    1. Procesar cada línea del fichero
    2. Añadir cada línea a la variable global del lexicón

In [22]:
def obtenerLexicon(rutaArchivo):
  # Comprobamos si existe el archivo pasado
  try:
    # Abrimos el archivo si es de extensión txt
        root, extension = os.path.splitext(rutaArchivo)
        if(extension == '.txt'):
          with open(rutaArchivo, "r", errors="ignore",encoding='utf-8-sig') as f:
            # Leemos cada línea/palabra del fichero
            lineaLeida = f.readline()
            while(lineaLeida):
              lineaProcesada = procesarTexto(lineaLeida)
              # Separamos la palabra por espacios
              arrayPalabras = lineaProcesada.split()

              linea = ""
              i = 0
              while(i < len(arrayPalabras)):
                linea += arrayPalabras[i]
                i += 1
                if(i < len(arrayPalabras)):
                  linea += " "

              global lexicon
              lexicon.add(linea)
              lineaLeida = f.readline()
     
          print("El número de palabras en el lexicón es de: " + str(len(lexicon)))
  except FileNotFoundError:
    print("No existe el archivo " + rutaArchivo)

## Función para entrenar el modelo
La función realizará los siguientes pasos:
1. Obtenemos el vector de características del los textos de entrenamiento.
2. Creamos el modelo SVM
3. Obtenemos todos los comentarios de entrenamiento y los procesamos
4. Obtenemos todos los términos del lexicón de insultos
5. Entrenamos el modelo con la función **.fit**

In [23]:
def entrenarModelo():
  # Obtenemos la lista del texto de comentarios
  listaDocumentos : list = []
  tags : list = []

  
  # Obtenemos todos los comentarios de entrenamiento
  global listaComentariosEntrenar
  for c in listaComentariosEntrenar:
    comentarioProcesado = procesarTexto(c.comment)
    listaDocumentos.append(comentarioProcesado)
    tags.append(c.label)
  
  '''
  # Obtenemos el lexicon procesado
  global lexicon
  for palabra in lexicon:
    listaDocumentos.append(palabra)
    tags.append('OFF')
  '''
  
  #Obtenemos el vector de características del los textos de entrenamiento que se han pasado
  global vectorizer
  vectorizer = TfidfVectorizer()
  X = vectorizer.fit_transform(listaDocumentos)
  # Creamos el modelo con la clase SVC
  global model
  model = svm.SVC()
  model.fit(X, tags) # Entrenamiento

## Función para predecir un texto
Función que recibe como parámetros:
* **documento** : Texto que se quiere predecir

La función realiza los siguientes pasos:
1. Procesamos el texto y lo almacenamos en una variable local.
2. Obtenemos el vector de características del texto a predecir
3. Predecimos a la etiqueta que pertenecerá el texto procesado

In [24]:
def predecirComentario(documento):
  # Procesamos el texto pasado
  contenido = [procesarTexto(documento)]
  
  # Obtenemos la predicción con el aprendizaje supervisado
  global vectorizer
  Y = vectorizer.transform(contenido)

  global model
  prediction = model.predict(Y)

  # Devolemos la predicción
  return prediction[0]

## Prediccion de comentarios
Función que recibe como parámetros
* **rutaFicheroPrediccion**: Ruta del fichero que almacenará los resultados de la predicción de la lista de comentarios para test  

La función realiza los siguientes pasos:
1. Crear el fichero en la ruta indicada como parámetro
2. Recorrer toda la lista de los comentarios de test
    1. Predecimos la categoría para cada comentario
    2. Escribimos el resultado en el fichero

In [25]:
def predecirComentarios(rutaFicheroPrediccion):
    global listaComentariosTest
    # Comprobamos que hay al menos un comentario en la lista de test
    if(len(listaComentariosTest) > 0):
      # Creamos un fichero
      ficheroPrediccion = open(rutaFicheroPrediccion, "w") #si existía el fichero, ha borrado su contenido
      ficheroPrediccion.write("comment_id\tpred\n")

      # Recorremos toda la lista de comentarios de test
      for c in listaComentariosTest:
        # Obtenemos el contenido del comentario
        prediccion = predecirComentario(c.comment)
        # Guardamos el resultado en el fichero
        linea = str(c.comment_id) + "\t" + str(prediccion)+ "\n"
        ficheroPrediccion.write(linea)

      # Cerramos el fichero
      ficheroPrediccion.close()

## Función para obtener el porcentaje de insultos a cada género
Función que recibe como parámetros:
* **listaComentario**: Lista de comentarios
* **rutaFicheroPrediccion**: Ruta del fichero de la predicción

In [26]:
def porcentajeGenero(listaComentarios,rutaFicheroPrediccion):
  man = 0
  woman = 0

  # Pasamos la lista de comentarios a diccionario por ID-GENERO
  diccionarioGenero: dict = {}
  for c in listaComentarios:
    diccionarioGenero[c.comment_id] = c.influencer_gender

  # Abrimos el fichero de comentarios predecidos
  try:
    # Abrimos el archivo 
    f = open(rutaFicheroPrediccion, "r", errors="ignore")

    # Separamos cada línea
    read_tsv = csv.reader(f, delimiter="\t")
    
    # Leemos todas las lineas 
    for r in read_tsv:
      # Comprobamos si el mensaje es ofensivo
      if(r[1] == "OFF"):
        # Buscamos en la lista de comentarios si se trata de un hombre o una mujer
        if(diccionarioGenero[r[0]] == "man"):
          man += 1
        else:
          woman += 1

    # Mostramos los resultados
    print("El porcentaje de mensajes ofensivos a hombres(man): " + str((man * 100) / (man + woman)) + " %" )
    print("El porcentaje de mensajes ofensivos a mujeres(woman): " + str((woman * 100) / (man + woman)) + " %" )

  except FileNotFoundError:
    print("No existe el archivo " + str(rutaFicheroPrediccion))

## Función para obtener el porcentaje de insultos en cada plataforma
Función que recibe como parámetros:
* **listaComentario**: Lista de comentarios
* **rutaFicheroPrediccion**: Ruta del fichero de la predicción

In [27]:
def porcentajePlataforma(listaComentarios,rutaFicheroPrediccion):
  conjuntoPlatafromas: set = set()
  diccionarioFinal: dict = {}

  # Pasamos la lista de comentarios a diccionario por ID-Plataforma
  diccionarioPlataforma: dict = {}
  for c in listaComentarios:
    diccionarioPlataforma[c.comment_id] = c.media
    if(c.media not in conjuntoPlatafromas):
      conjuntoPlatafromas.add(c.media)
      diccionarioFinal[c.media] = 0

  # Abrimos el fichero de comentarios predecidos
  try:
    # Abrimos el archivo 
    f = open(rutaFicheroPrediccion, "r", errors="ignore")

    # Separamos cada línea
    read_tsv = csv.reader(f, delimiter="\t")
    
    # Leemos todas las lineas
    total = 0
    for r in read_tsv:
      # Comprobamos si el mensaje es ofensivo
      if(r[1] == "OFF"):
        # Buscamos en la lista de comentarios para saber a que plataforma pertenece
        plataforma = diccionarioPlataforma[r[0]]
        diccionarioFinal[plataforma] += 1
        total += 1
    
    # Mostramos los resultados para cada plataforma
    for p in conjuntoPlatafromas:
      porcentaje = diccionarioFinal[p] * 100 / total
      print("El porcentaje de mensajes ofensivos en la plataforma " + p + " es de: " + str(porcentaje) + " %" )

  except FileNotFoundError:
    print("No existe el archivo " + str(rutaFicheroPrediccion))

## Ejecución del ejercicio
Para ejecutar el ejercicio, primero deberemos limpiar la lista de documentos y categorias y asignar a nulo el modelo y vector a entrenar.

In [28]:
# Obtenemos los comentarios de la lista de entrenamiento
listaComentariosEntrenar.clear()
obtenerDocumentosTSV("/content/drive/MyDrive/PLN/Practica_final/Material/train.tsv",listaComentariosEntrenar)
# Obtenemos los comentarios de la lista de test
listaComentariosTest.clear()
obtenerDocumentosTSV("/content/drive/MyDrive/PLN/Practica_final/Material/test.tsv",listaComentariosTest)
# Obtenemos el lexicon
lexicon.clear()
obtenerLexicon("/content/drive/MyDrive/PLN/Practica_final/Material/SHARE.txt")
# Entrenamos el modelo
model = None
vectorizer = None
entrenarModelo()

El número de comentarios es de: 16710
El número de comentarios es de: 13606
El número de palabras en el lexicón es de: 9133


In [29]:
# Predecimos el test
predecirComentarios("/content/drive/MyDrive/PLN/Practica_final/Material/prediction_test.tsv")


In [30]:
porcentajeGenero(listaComentariosTest,"/content/drive/MyDrive/PLN/Practica_final/Material/prediction_test.tsv")
print()
porcentajePlataforma(listaComentariosTest,"/content/drive/MyDrive/PLN/Practica_final/Material/prediction_test.tsv")

El porcentaje de mensajes ofensivos a hombres(man): 57.72870662460568 %
El porcentaje de mensajes ofensivos a mujeres(woman): 42.27129337539432 %

El porcentaje de mensajes ofensivos en la plataforma youtube es de: 76.76130389064143 %
El porcentaje de mensajes ofensivos en la plataforma twitter es de: 5.257623554153523 %
El porcentaje de mensajes ofensivos en la plataforma instagram es de: 17.981072555205046 %


## Fichero generado por el script scorer.py
Métricas que se utilizan:

1. **Precisión**:Con la métrica de precisión podemos medir la calidad del modelo de machine learning en tareas de clasificación.
2. **Recall**: La métrica de exhaustividad nos va a informar sobre la cantidad que el modelo de machine learning es capaz de identificar.
3. **F1_SCORE**: El valor F1 se utiliza para combinar las medidas de precision y recall en un sólo valor. Esto es práctico porque hace más fácil el poder comparar el rendimiento combinado de la precisión y la exhaustividad entre varias soluciones.


In [31]:
!python /content/drive/MyDrive/PLN/Practica_final/Material/scorer.py /content/drive/MyDrive/PLN/Practica_final/Material/gold_label_test.tsv /content/drive/MyDrive/PLN/Practica_final/Material/prediction_test.tsv

GOLD_FILE: /content/drive/MyDrive/PLN/Practica_final/Material/gold_label_test.tsv
PREDICTION_FILE: /content/drive/MyDrive/PLN/Practica_final/Material/prediction_test.tsv
/usr/local/lib/python3.7/dist-packages/numpy/lib/arraysetops.py:604: FutureWarning: elementwise comparison failed; returning scalar instead, but in the future will perform elementwise comparison
  mask &= (ar1 != a)
/usr/local/lib/python3.7/dist-packages/sklearn/metrics/_classification.py:1318: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/usr/local/lib/python3.7/dist-packages/numpy/lib/arraysetops.py:604: FutureWarning: elementwise comparison failed; returning scalar instead, but in the future will perform elementwise comparison
  mask &= (ar1 != a)
/usr/local/lib/python3.7/dist-packages/sklearn/metrics/_classification.py:1318: UndefinedMetricWarni